In [1]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

e:\Agentic AI\VoyageAI Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Please run 01_rag_ingestion.ipynb first.")

Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8668.00it/s]


Embedding model loaded successfully.


In [4]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("Chroma vector database loaded successfully.")
print("Total vectors available:", vector_store._collection.count())

Chroma vector database loaded successfully.
Total vectors available: 45


In [5]:
def get_retrieval_context(query: str, k: int = 3):
    docs = vector_store.similarity_search(query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [7]:
context = get_retrieval_context(
    query="I want scuba diving, clean beaches and island hopping",
    k=3
)

print(context)

Context 1
City: andaman
Source: andaman.txt
Content:
Destination: Andaman and Nicobar Islands
State/Region: Union Territory
Destination Type: Islands, beaches, scuba diving, snorkeling, honeymoon, water sports

Overview:
Andaman is known for clean beaches, blue waters, coral reefs, scuba diving, snorkeling, island hopping, and peaceful tropical experiences.

Best For:
- Beach lovers
- Honeymoon couples
- Scuba diving beginners
- Snorkeling lovers
- Nature lovers
- Luxury island travelers

Context 2
City: andaman
Source: andaman.txt
Content:
Top Attractions:
- Radhanagar Beach
- Havelock Island
- Neil Island
- Cellular Jail
- North Bay Island
- Ross Island
- Elephant Beach
- Bharatpur Beach

Food Recommendations:
- Seafood
- Fish curry
- Prawn dishes
- Crab dishes
- Coconut-based coastal food
- Indian and continental island cafes

Stay Suggestions:
- Budget: simple guesthouses in Port Blair or Havelock
- Comfort: beachside cottages
- Luxury: premium island resorts

Context 3
City: pondi